# TP 1 — LDA/QDA y optimización matemática de modelos

## Integrantes
- Jonatan Mild
- Valentin Torres
- Victor Astorga
- Franco Morero
- Francisco Meaca

**Materia:** Análisis Matemático para Inteligencia Artificial | **Año:** 2026



## Versiones utilizadas

In [1]:
import sys
import numpy
import scipy

print(f"Python {sys.version}")
print(f"NumPy  {numpy.__version__}")
print(f"SciPy  {scipy.__version__}")

Python 3.13.1 (tags/v3.13.1:0671451, Dec  3 2024, 19:06:28) [MSC v.1942 64 bit (AMD64)]
NumPy  2.4.4
SciPy  1.17.1


## Código base


In [2]:
import numpy as np
import pandas as pd
import numpy.linalg as LA
from scipy.linalg import cholesky, solve_triangular
from scipy.linalg.lapack import dtrtri


In [3]:
class BaseBayesianClassifier:
  def __init__(self):
    pass

  def _estimate_a_priori(self, y):
    a_priori = np.bincount(y.flatten().astype(int)) / y.size
    return np.log(a_priori)

  def _fit_params(self, X, y):
    # estimate all needed parameters for given model
    raise NotImplementedError()

  def _predict_log_conditional(self, x, class_idx):
    # predict the log(P(x|G=class_idx)), the log of the conditional probability of x given the class
    # this should depend on the model used
    raise NotImplementedError()

  def fit(self, X, y, a_priori=None):
    # if it's needed, estimate a priori probabilities
    self.log_a_priori = self._estimate_a_priori(y) if a_priori is None else np.log(a_priori)

    # now that everything else is in place, estimate all needed parameters for given model
    self._fit_params(X, y)

  def predict(self, X):
    # this is actually an individual prediction encased in a for-loop
    m_obs = X.shape[1]
    y_hat = np.empty(m_obs, dtype=int)

    for i in range(m_obs):
      y_hat[i] = self._predict_one(X[:,i].reshape(-1,1))

    # return prediction as a row vector (matching y)
    return y_hat.reshape(1,-1)

  def _predict_one(self, x):
    # calculate all log posteriori probabilities (actually, +C)
    log_posteriori = [ log_a_priori_i + self._predict_log_conditional(x, idx) for idx, log_a_priori_i
                  in enumerate(self.log_a_priori) ]

    # return the class that has maximum a posteriori probability
    return np.argmax(log_posteriori)

In [4]:
class QDA(BaseBayesianClassifier):

  def _fit_params(self, X, y):
    # estimate each covariance matrix
    self.inv_covs = [LA.inv(np.cov(X[:,y.flatten()==idx], bias=True))
                      for idx in range(len(self.log_a_priori))]
    self.means = [X[:,y.flatten()==idx].mean(axis=1, keepdims=True)
                  for idx in range(len(self.log_a_priori))]

  def _predict_log_conditional(self, x, class_idx):
    # predict the log(P(x|G=class_idx)), the log of the conditional probability of x given the class
    # this should depend on the model used
    inv_cov = self.inv_covs[class_idx]
    unbiased_x =  x - self.means[class_idx]
    return 0.5*np.log(LA.det(inv_cov)) -0.5 * unbiased_x.T @ inv_cov @ unbiased_x

In [5]:
class TensorizedQDA(QDA):

    def _fit_params(self, X, y):
        # ask plain QDA to fit params
        super()._fit_params(X,y)

        # stack onto new dimension
        self.tensor_inv_cov = np.stack(self.inv_covs)
        self.tensor_means = np.stack(self.means)

    def _predict_log_conditionals(self,x):
        unbiased_x = x - self.tensor_means
        inner_prod = unbiased_x.transpose(0,2,1) @ self.tensor_inv_cov @ unbiased_x

        return 0.5*np.log(LA.det(self.tensor_inv_cov)) - 0.5 * inner_prod.flatten()

    def _predict_one(self, x):
        # return the class that has maximum a posteriori probability
        return np.argmax(self.log_a_priori + self._predict_log_conditionals(x))

In [6]:
class QDA_Chol1(BaseBayesianClassifier):
  def _fit_params(self, X, y):
    self.L_invs = [
        LA.inv(cholesky(np.cov(X[:,y.flatten()==idx], bias=True), lower=True))
        for idx in range(len(self.log_a_priori))
    ]

    self.means = [X[:,y.flatten()==idx].mean(axis=1, keepdims=True)
                  for idx in range(len(self.log_a_priori))]

  def _predict_log_conditional(self, x, class_idx):
    L_inv = self.L_invs[class_idx]
    unbiased_x =  x - self.means[class_idx]

    y = L_inv @ unbiased_x

    return np.log(L_inv.diagonal().prod()) -0.5 * (y**2).sum()

In [7]:
class QDA_Chol2(BaseBayesianClassifier):
  def _fit_params(self, X, y):
    self.Ls = [
        cholesky(np.cov(X[:,y.flatten()==idx], bias=True), lower=True)
        for idx in range(len(self.log_a_priori))
    ]

    self.means = [X[:,y.flatten()==idx].mean(axis=1, keepdims=True)
                  for idx in range(len(self.log_a_priori))]

  def _predict_log_conditional(self, x, class_idx):
    L = self.Ls[class_idx]
    unbiased_x =  x - self.means[class_idx]

    y = solve_triangular(L, unbiased_x, lower=True)

    return -np.log(L.diagonal().prod()) -0.5 * (y**2).sum()

In [8]:
class QDA_Chol3(BaseBayesianClassifier):
  def _fit_params(self, X, y):
    self.L_invs = [
        dtrtri(cholesky(np.cov(X[:,y.flatten()==idx], bias=True), lower=True), lower=1)[0]
        for idx in range(len(self.log_a_priori))
    ]

    self.means = [X[:,y.flatten()==idx].mean(axis=1, keepdims=True)
                  for idx in range(len(self.log_a_priori))]

  def _predict_log_conditional(self, x, class_idx):
    L_inv = self.L_invs[class_idx]
    unbiased_x =  x - self.means[class_idx]

    y = L_inv @ unbiased_x

    return np.log(L_inv.diagonal().prod()) -0.5 * (y**2).sum()

## Datasets

In [9]:
from sklearn.datasets import load_iris, fetch_openml, load_wine
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

def get_iris_dataset():
  data = load_iris()
  X_full = data.data
  y_full = np.array([data.target_names[y] for y in data.target.reshape(-1,1)])
  return X_full, y_full

def get_penguins_dataset():
    # get data
    df, tgt = fetch_openml(name="penguins", return_X_y=True, as_frame=True, parser='auto')

    # drop non-numeric columns
    df.drop(columns=["island","sex"], inplace=True)

    # drop rows with missing values
    mask = df.isna().sum(axis=1) == 0
    df = df[mask]
    tgt = tgt[mask]

    return df.values, tgt.to_numpy().reshape(-1,1)

def get_wine_dataset():
    # get data
    data = load_wine()
    X_full = data.data
    y_full = np.array([data.target_names[y] for y in data.target.reshape(-1,1)])
    return X_full, y_full

def get_letters_dataset():
    # get data
    letter = fetch_openml('letter', version=1, as_frame=False)
    return letter.data, letter.target.reshape(-1,1)

def label_encode(y_full):
    return LabelEncoder().fit_transform(y_full.flatten()).reshape(y_full.shape)

def split_transpose(X, y, test_size, random_state):
    # X_train, X_test, y_train, y_test but all transposed
    return [elem.T for elem in train_test_split(X, y, test_size=test_size, random_state=random_state)]

## Benchmarking

In [10]:
import time
from tqdm.notebook import tqdm
from numpy.random import RandomState
import tracemalloc

RNG_SEED = 6553

class Benchmark:
    def __init__(self, X, y, n_runs=1000, warmup=100, mem_runs=100, test_sz=0.3, rng_seed=RNG_SEED, same_splits=True):
        self.X = X
        self.y = y
        self.n = n_runs
        self.warmup = warmup
        self.mem_runs = mem_runs
        self.test_sz = test_sz
        self.det = same_splits
        if self.det:
            self.rng_seed = rng_seed
        else:
            self.rng = RandomState(rng_seed)

        self.data = dict()

        print("Benching params:")
        print("Total runs:",self.warmup+self.mem_runs+self.n)
        print("Warmup runs:",self.warmup)
        print("Peak Memory usage runs:", self.mem_runs)
        print("Running time runs:", self.n)
        approx_test_sz = int(self.y.size * self.test_sz)
        print("Train size rows (approx):",self.y.size - approx_test_sz)
        print("Test size rows (approx):",approx_test_sz)
        print("Test size fraction:",self.test_sz)

    def bench(self, model_class, **kwargs):
        name = model_class.__name__
        time_data = np.empty((self.n, 3), dtype=float)  # train_time, test_time, accuracy
        mem_data = np.empty((self.mem_runs, 2), dtype=float)  # train_peak_mem, test_peak_mem
        rng = RandomState(self.rng_seed) if self.det else self.rng


        for i in range(self.warmup):
            # Instantiate model with error check for unsupported parameters
            model = model_class(**kwargs)

            # Generate current train-test split
            X_train, X_test, y_train, y_test = split_transpose(
                self.X, self.y,
                test_size=self.test_sz,
                random_state=rng
            )
            # Run training and prediction (timing or memory measurement not recorded)
            model.fit(X_train, y_train)
            model.predict(X_test)

        for i in tqdm(range(self.mem_runs), total=self.mem_runs, desc=f"{name} (MEM)"):

            model = model_class(**kwargs)

            X_train, X_test, y_train, y_test = split_transpose(
                self.X, self.y,
                test_size=self.test_sz,
                random_state=rng
            )

            tracemalloc.start()

            t1 = time.perf_counter()
            model.fit(X_train, y_train)
            t2 = time.perf_counter()

            _, train_peak = tracemalloc.get_traced_memory()
            tracemalloc.reset_peak()

            model.predict(X_test)
            t3 = time.perf_counter()
            _, test_peak = tracemalloc.get_traced_memory()
            tracemalloc.stop()

            mem_data[i,] = (
                train_peak / (1024 * 1024),
                test_peak / (1024 * 1024)
            )

        for i in tqdm(range(self.n), total=self.n, desc=f"{name} (TIME)"):
            model = model_class(**kwargs)

            X_train, X_test, y_train, y_test = split_transpose(
                self.X, self.y,
                test_size=self.test_sz,
                random_state=rng
            )

            t1 = time.perf_counter()
            model.fit(X_train, y_train)
            t2 = time.perf_counter()
            preds = model.predict(X_test)
            t3 = time.perf_counter()

            time_data[i,] = (
                (t2 - t1) * 1000,
                (t3 - t2) * 1000,
                (y_test.flatten() == preds.flatten()).mean()
            )

        self.data[name] = (time_data, mem_data)

    def summary(self, baseline=None):
        aux = []
        for name, (time_data, mem_data) in self.data.items():
            result = {
                'model': name,
                'train_median_ms': np.median(time_data[:, 0]),
                'train_std_ms': time_data[:, 0].std(),
                'test_median_ms': np.median(time_data[:, 1]),
                'test_std_ms': time_data[:, 1].std(),
                'mean_accuracy': time_data[:, 2].mean(),
                'train_mem_median_mb': np.median(mem_data[:, 0]),
                'train_mem_std_mb': mem_data[:, 0].std(),
                'test_mem_median_mb': np.median(mem_data[:, 1]),
                'test_mem_std_mb': mem_data[:, 1].std()
            }
            aux.append(result)
        df = pd.DataFrame(aux).set_index('model')

        if baseline is not None and baseline in self.data:
            df['train_speedup'] = df.loc[baseline, 'train_median_ms'] / df['train_median_ms']
            df['test_speedup'] = df.loc[baseline, 'test_median_ms'] / df['test_median_ms']
            df['train_mem_reduction'] = df.loc[baseline, 'train_mem_median_mb'] / df['train_mem_median_mb']
            df['test_mem_reduction'] = df.loc[baseline, 'test_mem_median_mb'] / df['test_mem_median_mb']
        return df

# Consigna QDA

**Notación**: en general notamos

* $k$ la cantidad de clases
* $n$ la cantidad de observaciones
* $p$ la cantidad de features/variables/predictores


## Tensorización


### 1) Diferencias entre `QDA`y `TensorizedQDA`

1. ¿Sobre qué paraleliza `TensorizedQDA`? ¿Sobre las $k$ clases, las $n$ observaciones a predecir, o ambas?

El modelo `QDA` base implementa la predicción de una manera secuencial. Para una observación dada, representada por el vector $\mathbf{x}$, recorre cada una de las $k$ clases para calcular la función discriminante $\delta_j(\mathbf{x})$. Este proceso se repite para cada una de las $n$ observaciones en el conjunto de prueba.

`TensorizedQDA`, en cambio, aprovecha las capacidades de cómputo vectorial de NumPy para optimizar uno de estos procesos. En lugar de iterar sobre las $k$ clases, apila los parámetros de cada clase (las medias $\boldsymbol{\mu}_j$ y las inversas de las covarianzas $\mathbf{\Sigma}_j^{-1}$) en estructuras de datos de mayor dimensión, conocidas como tensores. Esto le permite calcular las $k$ funciones discriminantes para una única observación $\mathbf{x}$ de manera simultánea, a través de operaciones de broadcasting y multiplicación de matrices por lotes (batched matrix multiplication).

Sin embargo, es crucial notar que `TensorizedQDA` aún procesa las observaciones una por una. El método `predict` de la clase base, que contiene el bucle `for` sobre las $n$ observaciones, no se modifica. Por lo tanto, podemos afirmar que:

**`TensorizedQDA` paraleliza el cálculo sobre las $k$ clases, pero no sobre las $n$ observaciones a predecir.**

2. Analizar los shapes de `tensor_inv_covs` y `tensor_means` y explicar paso a paso cómo es que `TensorizedQDA` llega a predecir lo mismo que `QDA`.


---

#### **Punto 2: Análisis de shapes y lógica de `TensorizedQDA`**

Para entender cómo `TensorizedQDA` logra el mismo resultado que `QDA` sin un bucle explícito sobre las clases, primero debemos definir la notación y los parámetros involucrados.

*   $k$: Número de clases.
*   $p$: Número de features (dimensiones).
*   $\boldsymbol{\mu}_j \in \mathbb{R}^{p \times 1}$: Vector de medias para la clase $j$.
*   $\mathbf{\Sigma}_j \in \mathbb{R}^{p \times p}$: Matriz de covarianza para la clase $j$.
*   $\mathbf{x} \in \mathbb{R}^{p \times 1}$: Una observación a clasificar.

En la etapa de `fit`, el modelo calcula las listas de estos parámetros. `TensorizedQDA` luego las convierte a tensores:

*   `self.means` es una lista de $k$ vectores de forma `(p, 1)`. Al apilarlos, `tensor_means` se convierte en un tensor $\boldsymbol{\mathcal{M}}$ de forma `(k, p, 1)`.
*   `self.inv_covs` es una lista de $k$ matrices de forma `(p, p)`. Al apilarlos, `tensor_inv_cov` se convierte en un tensor $\mathbf{\mathcal{S}}^{-1}$ de forma `(k, p, p)`.

Ahora, analicemos el método `_predict_log_conditionals` paso a paso para una observación $\mathbf{x}$:

1.  **Cálculo de las diferencias (centrado):**
    El primer paso es centrar la observación $\mathbf{x}$ con respecto a la media de cada clase. En lugar de hacerlo en un bucle, se aprovecha el broadcasting de NumPy. Restamos el tensor de medias $\boldsymbol{\mathcal{M}}$ (forma `(k, p, 1)`) del vector $\mathbf{x}$ (forma `(p, 1)`). NumPy "expande" $\mathbf{x}$ para que coincida con la forma de $\boldsymbol{\mathcal{M}}$, realizando $k$ restas en una sola operación. El resultado es un tensor que contiene todos los vectores de diferencia.

    $$
    \underset{(k, p, 1)}{\mathbf{\Delta}} = \underset{(p, 1)}{\mathbf{x}} - \underset{(k, p, 1)}{\boldsymbol{\mathcal{M}}}
    $$

    Donde la $j$-ésima "rebanada" (slice) de $\mathbf{\Delta}$ es el vector $\boldsymbol{\delta}_j = \mathbf{x} - \boldsymbol{\mu}_j$.

2.  **Cálculo de la forma cuadrática:**
    El siguiente paso es calcular la distancia de Mahalanobis al cuadrado para cada clase, que es la forma cuadrática $\boldsymbol{\delta}_j^T \mathbf{\Sigma}_j^{-1} \boldsymbol{\delta}_j$. NumPy realiza esta operación para las $k$ clases simultáneamente. El producto `@` entre tensores efectúa una multiplicación de matrices por lotes en los dos últimos ejes.

    $$
    \underset{(k, 1, 1)}{\mathbf{Q}} = \underset{(k, 1, p)}{\mathbf{\Delta}^T} \ @ \ \underset{(k, p, p)}{\mathbf{\mathcal{S}}^{-1}} \ @ \ \underset{(k, p, 1)}{\mathbf{\Delta}}
    $$

    El resultado $\mathbf{Q}$ es un tensor donde la $j$-ésima rebanada contiene el valor escalar de la forma cuadrática para la clase $j$. El método `.flatten()` lo convierte en un vector de forma `(k,)`.

3.  **Cálculo del término del determinante:**
    Finalmente, el término log-determinante se calcula de manera similar. La función `np.linalg.det` aplicada a un tensor de forma `(k, p, p)` devuelve un vector de forma `(k,)` que contiene el determinante de cada una de las $k$ matrices.

    $$
    \underset{(k,)}{\mathbf{d}} = \text{det}(\underset{(k, p, p)}{\mathbf{\mathcal{S}}^{-1}})
    $$

Al combinar estos componentes, el método calcula el vector completo de log-verosimilitudes condicionales, una para cada clase, sin necesidad de un bucle `for`.

### Optimización

3. Implementar el modelo `FasterQDA` de manera de eliminar el ciclo for en el método predict.

In [11]:
# FasterQDA elimina el for-loop sobre observaciones de predict.
# El flujo es:
#   1. Centrar: U = X - μ_j → (k, p, n) por broadcasting
#   2. Producto: Σ_j⁻¹ U → (k, p, n)
#   3. Forma cuadrática: Uᵀ Σ⁻¹ U → (k, n, n) — acá aparece la matriz n×n
#   4. Diagonal: se extrae con np.diagonal → (k, n) — solo las distancias²
#   5. Log-posterior: suma log-prior + log-determinante - ½ distancias² → (k, n)
#   6. Argmax: por columna (axis=0) → clase ganadora para cada observación

class FasterQDA(TensorizedQDA):
    def _predict_log_conditionals(self, X):
        # X shape: (p, n), tensor_means shape: (k, p, 1), tensor_inv_cov shape: (k, p, p)
        # 1. Centrar X respecto a cada clase: (p, n) - (k, p, 1) → (k, p, n)

        U = X - self.tensor_means

         # 2. Producto Σ_j⁻¹ @ U para cada clase: (k, p, p) @ (k, p, n) → (k, p, n)

        SU = self.tensor_inv_cov @ U

        # 3. Forma cuadrática Uᵀ Σ⁻¹ U por clase: (k, n, p) @ (k, p, n) → (k, n, n)

        quad = U.transpose(0, 2, 1) @ SU

        # 4. Extraer diagonal (dist² de cada obs): (k, n, n) → (k, n)

        dist = np.diagonal(quad, axis1=1, axis2=2)

         # 5. Log-posterior por clase: (k, 1) - (k, n) → (k, n)

        log_det = 0.5 * np.log(LA.det(self.tensor_inv_cov)).reshape(-1, 1)


        return log_det - 0.5 * dist


    def predict(self, X):
        
        
       
        log_post = (self.log_a_priori.reshape(-1, 1) + self._predict_log_conditionals(X))

        # 6. Clase con máximo a posteriori para cada obs: (n,) → (1, n)
        return np.argmax(log_post, axis=0).reshape(1, -1)

4. Mostrar dónde aparece la mencionada matriz de $n \times n$, donde $n$ es la cantidad de observaciones a predecir.


La matriz de $n \times n$ emerge cuando intentamos extender la paralelización del cálculo de la forma cuadrática para que no solo abarque las $k$ clases, sino también las $n$ observaciones del conjunto de prueba simultáneamente.

Consideremos el cálculo para una sola clase $j$. Si en lugar de una única observación $\mathbf{x} \in \mathbb{R}^{p \times 1}$, queremos procesar el conjunto completo de $n$ observaciones, que representamos como una matriz $\mathbf{X} \in \mathbb{R}^{p \times n}$:

1.  **Centrado del conjunto de datos:**
    Primero, centramos cada columna (observación) de $\mathbf{X}$ restando el vector de medias $\boldsymbol{\mu}_j$. Esto se logra mediante broadcasting:

    $$
    \underset{(p, n)}{\mathbf{U}_j} = \underset{(p, n)}{\mathbf{X}} - \underset{(p, 1)}{\boldsymbol{\mu}_j}
    $$

2.  **Cálculo de la forma cuadrática matricial:**
    La forma cuadrática para el conjunto de datos completo se calcula como:

    $$
    \underset{(n, n)}{\mathbf{Q}_j} = \underset{(n, p)}{\mathbf{U}_j^T} \ \underset{(p, p)}{\mathbf{\Sigma}_j^{-1}} \ \underset{(p, n)}{\mathbf{U}_j}
    $$

Aquí es donde se materializa la matriz de $n \times n$. Cada elemento $(i, m)$ de esta matriz $\mathbf{Q}_j$ representa el producto cruzado $(\mathbf{x}_i - \boldsymbol{\mu}_j)^T \mathbf{\Sigma}_j^{-1} (\mathbf{x}_m - \boldsymbol{\mu}_j)$. Para la función discriminante, solo nos interesan los elementos de la diagonal, donde $i=m$, que corresponden a la distancia de Mahalanobis al cuadrado para cada observación. Construir la matriz completa es computacionalmente costoso y un derroche de memoria, ya que la mayor parte de la información calculada (los elementos fuera de la diagonal) se descarta.


5. Demostrar que
$$
diag(A \cdot B) = \sum_{cols} A \odot B^T = np.sum(A \odot B^T, axis=1)
$$ es decir, que se puede "esquivar" la matriz de $n \times n$ usando matrices de $n \times p$. También se puede usar, de forma equivalente,
$$
np.sum(A^T \odot B, axis=0).T
$$
queda a preferencia del alumno cuál usar.

**Demostración:**

Por definición, el elemento $(i, j)$ de la matriz producto $\mathbf{C} = \mathbf{A}\mathbf{B}$ se calcula como:

$$
C_{ij} = (\mathbf{A}\mathbf{B})_{ij} = \sum_{k=1}^{p} A_{ik} B_{kj}
$$

Nos interesan los elementos de la diagonal de $\mathbf{C}$, que son aquellos donde $i=j$:

$$
C_{ii} = (\mathbf{A}\mathbf{B})_{ii} = \sum_{k=1}^{p} A_{ik} B_{ki}
$$

Ahora, consideremos la segunda parte de la proposición. La transpuesta de $\mathbf{B}$ es $\mathbf{B}^T \in \mathbb{R}^{n \times p}$, donde el elemento $(i, k)$ es $(B^T)_{ik} = B_{ki}$.

El producto de Hadamard (o elemento a elemento) de $\mathbf{A}$ y $\mathbf{B}^T$, que denotamos $\mathbf{A} \odot \mathbf{B}^T$, es una matriz $\mathbf{H} \in \mathbb{R}^{n \times p}$ cuyo elemento $(i, k)$ es:

$$
H_{ik} = (\mathbf{A} \odot \mathbf{B}^T)_{ik} = A_{ik} \cdot (B^T)_{ik} = A_{ik} \cdot B_{ki}
$$

Si ahora sumamos los elementos de cada fila de $\mathbf{H}$ (es decir, sumamos sobre el índice de las columnas, $k$), obtenemos un vector cuyos componentes $i$ son:

$$
\left( \sum_{\text{cols}} \mathbf{H} \right)_i = \sum_{k=1}^{p} H_{ik} = \sum_{k=1}^{p} A_{ik} \cdot B_{ki}
$$

Este resultado es idéntico al elemento $C_{ii}$ de la diagonal de $\mathbf{A}\mathbf{B}$. Por lo tanto, hemos demostrado la igualdad.

La ventaja computacional es inmensa: en lugar de realizar una multiplicación matricial con un costo de $O(n^2 p)$ y almacenar una matriz intermedia de $O(n^2)$, realizamos un producto de Hadamard y una suma con un costo y almacenamiento de solo $O(np)$.

6. Utilizar la propiedad antes demostrada para reimplementar la predicción del modelo `FasterQDA` de forma eficiente en un nuevo modelo `EfficientQDA`.

In [12]:
# EfficientQDA usa diag(A·B) = sum(A ⊙ B^T, axis=1)
# para evitar construir la matriz n×n.
#
# En FasterQDA el cuello de botella era:
#   quad = U^T @ SU  → (k, n, n)   ← matriz n×n por clase
#   dist = diagonal(quad)          ← se descarta casi todo
#
# Aplicando la propiedad con A = U^T (k,n,p) y B = SU (k,p,n):
#   diag(U^T @ SU) = sum(U^T ⊙ SU^T, axis=1)
# Equivalentemente, trabajando en (k,p,n):
#   diag = sum(U * SU, axis=1)  → (k, n)  directamente
#
# Nunca se construye la matriz (k, n, n). Memoria: O(np) en vez de O(n²).

class EfficientQDA(TensorizedQDA):
    def _predict_log_conditionals(self, X):
        # X: (p, n), tensor_means: (k, p, 1), tensor_inv_cov: (k, p, p)

        # Centrar: (p, n) - (k, p, 1) → (k, p, n)
        U = X - self.tensor_means

        # Producto Σ_j⁻¹ @ U: (k, p, p) @ (k, p, n) → (k, p, n)
        SU = self.tensor_inv_cov @ U

        # diag(Uᵀ @ SU) = sum(U * SU, axis=1) → (k, n)
        # U y SU tienen shape (k, p, n), el * es element-wise,
        # y sum(axis=1) colapsa la dimensión p → producto punto por observación

        dist = np.sum(U * SU, axis=1)  # (k, n)

        # Log-posteriori: (k, 1) - (k, n) → (k, n)
        log_det = 0.5 * np.log(LA.det(self.tensor_inv_cov)).reshape(-1, 1)
        return log_det - 0.5 * dist

    def predict(self, X):


        log_post = (self.log_a_priori.reshape(-1, 1) + self._predict_log_conditionals(X))
        

        # Clase ganadora por observación: (n,) → (1, n)
        return np.argmax(log_post, axis=0).reshape(1, -1)

7. Comparar la performance de las 4 variantes de QDA implementadas hasta ahora (no Cholesky) ¿Qué se observa? A modo de opinión ¿Se condice con lo esperado?

In [13]:
# Benchmark de las 4 variantes con Letters dataset
X_letter, y_letter = get_letters_dataset()
y_letter_encoded = label_encode(y_letter.reshape(-1,1))

b_letter = Benchmark(
    X_letter, y_letter_encoded,
    same_splits=False,
    n_runs=100,
    warmup=20,
    mem_runs=30,
    test_sz=0.2
)

to_bench = [QDA, TensorizedQDA, FasterQDA, EfficientQDA]

for model in to_bench:
    b_letter.bench(model)

summ_letter = b_letter.summary(baseline='QDA')
summ_letter[[
    'train_median_ms', 'test_median_ms', 'mean_accuracy',
    'train_speedup', 'test_speedup',
    'train_mem_reduction', 'test_mem_reduction'
]]

Benching params:
Total runs: 150
Warmup runs: 20
Peak Memory usage runs: 30
Running time runs: 100
Train size rows (approx): 16000
Test size rows (approx): 4000
Test size fraction: 0.2


QDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

FasterQDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

FasterQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

EfficientQDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

EfficientQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

,train_median_ms,test_median_ms,mean_accuracy,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,
QDA,6.24885,1291.29160,0.886117,1.000000,1.000000,1.000000,1.000000
TensorizedQDA,5.86510,227.98145,0.885303,1.065429,5.664020,1.006077,0.641254
FasterQDA,6.45300,1669.60515,0.884827,0.968364,0.773411,1.004252,0.000031
EfficientQDA,6.22530,20.41040,0.884890,1.003783,63.266354,1.002433,0.002537


**Análisis de resultados (Punto 7):**

Los resultados del benchmark permiten observar con claridad el impacto de cada nivel de optimización sobre la performance en predicción.

**Tiempo de entrenamiento:** las cuatro variantes presentan tiempos de entrenamiento prácticamente idénticos (entre 5.87 ms y 6.45 ms). Esto es esperable, ya que `TensorizedQDA`, `FasterQDA` y `EfficientQDA` heredan el mismo `_fit_params` de `QDA` y no modifican la etapa de ajuste. Las diferencias observadas son ruido de medición y no reflejan ninguna diferencia algorítmica real.

**Tiempo de predicción — `TensorizedQDA`:** obtiene un speedup de ~5.66× respecto a `QDA` (de 1291 ms a 228 ms). Eliminar el bucle explícito sobre las $k = 26$ clases mediante broadcasting tensorial produce una aceleración clara y proporcional al número de clases.

**Tiempo de predicción — `FasterQDA`:** lejos de mejorar, resulta un 23% más lento que `QDA` base (speedup de 0.77×, de 1291 ms a 1670 ms). La causa es precisa: al construir la matriz $\mathbf{Q}_j = \mathbf{U}_j^T \mathbf{\Sigma}_j^{-1} \mathbf{U}_j \in \mathbb{R}^{4000 \times 4000}$ para cada una de las 26 clases, el overhead de alocación y escritura de memoria domina completamente el tiempo de cómputo. La reducción de memoria en test de $3.1 \times 10^{-5}$ (es decir, `FasterQDA` usa aproximadamente **32.000 veces más memoria** que `QDA`) confirma sin ambigüedades este diagnóstico. El modelo es matemáticamente correcto pero computacionalmente inviable a esta escala.

**Tiempo de predicción — `EfficientQDA`:** el resultado es sobresaliente: **63× más rápido en test** que `QDA` (de 1291 ms a 20 ms). La identidad $\text{diag}(\mathbf{A}\mathbf{B}) = \texttt{np.sum}(\mathbf{A} \odot \mathbf{B}^T, \texttt{axis}=1)$ no solo elimina la costosa matriz $n \times n$, sino que al reducir las operaciones a productos Hadamard sobre matrices de tamaño $\mathcal{O}(np)$ (con $p = 16$ fijo), permite que NumPy opere en regiones donde la localidad de caché es máxima. La reducción de uso de memoria en test (factor $\approx 400\times$ menos que `QDA`) es coherente con la eliminación de las 26 matrices de $4000 \times 4000$.

**Accuracy:** esencialmente idéntica en los cuatro modelos (0.8848–0.8861). Las pequeñas diferencias son producto de la variabilidad del split aleatorio, no de diferencias en las predicciones. Las implementaciones son matemáticamente equivalentes.

**Conclusión:** la secuencia `QDA` → `TensorizedQDA` → `EfficientQDA` representa una cadena de dos optimizaciones ortogonales y acumulables: tensorizar sobre clases (~ 5.7×) y vectorizar sobre observaciones sin construir la matriz $n \times n$ (~11× adicional), resultando en una aceleración combinada de ~63×. `FasterQDA` ilustra por contraste que vectorizar sobre observaciones *sin* evitar la matriz $n \times n$ es no solo ineficiente sino contraproducente frente a la implementación base.

### Diferencias entre implementaciones de `QDA_Chol`

8. Si una matriz $A$ tiene fact. de Cholesky $A=LL^T$, expresar $A^{-1}$ en términos de $L$. ¿Cómo podría esto ser útil en la forma cuadrática de QDA?


Dada una matriz simétrica y definida positiva $\mathbf{A}$, su descomposición de Cholesky es $\mathbf{A} = \mathbf{L}\mathbf{L}^T$, donde $\mathbf{L}$ es una matriz triangular inferior.

Para encontrar la inversa de $\mathbf{A}$, invertimos ambos lados de la ecuación:

$$
\mathbf{A}^{-1} = (\mathbf{L}\mathbf{L}^T)^{-1}
$$

Utilizando la propiedad de la inversa de un producto de matrices, $(XY)^{-1} = Y^{-1}X^{-1}$, obtenemos:

$$
\mathbf{A}^{-1} = (\mathbf{L}^T)^{-1} \mathbf{L}^{-1}
$$

Además, como la inversa de la transpuesta es la transpuesta de la inversa, $(\mathbf{L}^T)^{-1} = (\mathbf{L}^{-1})^T$, podemos reescribir la expresión como:

$$
\mathbf{A}^{-1} = (\mathbf{L}^{-1})^T \mathbf{L}^{-1}
$$

Esta formulación es extremadamente útil en la forma cuadrática de QDA. La expresión original $(\mathbf{x}-\boldsymbol{\mu})^T \mathbf{\Sigma}^{-1} (\mathbf{x}-\boldsymbol{\mu})$ se transforma. Sustituyendo $\mathbf{\Sigma}^{-1} = (\mathbf{L}^{-1})^T \mathbf{L}^{-1}$, tenemos:

$$
(\mathbf{x}-\boldsymbol{\mu})^T (\mathbf{L}^{-1})^T \mathbf{L}^{-1} (\mathbf{x}-\boldsymbol{\mu})
$$

Agrupando términos, esto es equivalente a:

$$
(\mathbf{L}^{-1}(\mathbf{x}-\boldsymbol{\mu}))^T (\mathbf{L}^{-1}(\mathbf{x}-\boldsymbol{\mu}))
$$

Si definimos un nuevo vector $\mathbf{y} = \mathbf{L}^{-1}(\mathbf{x}-\boldsymbol{\mu})$, la forma cuadrática se simplifica a $\mathbf{y}^T\mathbf{y}$, que no es más que la norma euclidiana al cuadrado de $\mathbf{y}$, $\|\mathbf{y}\|_2^2$. Esto reemplaza una multiplicación matriz-vector-matriz por una multiplicación matriz-vector seguida de un producto punto, lo cual es más eficiente y numéricamente más estable.

9. Explicar las diferencias entre `QDA_Chol1`y `QDA` y cómo `QDA_Chol1` llega, paso a paso, hasta las predicciones.


La principal diferencia radica en cómo se calcula y utiliza la inversa de la matriz de covarianza.

*   **`QDA`:** Calcula la matriz de covarianza $\mathbf{\Sigma}_j$ y luego su inversa $\mathbf{\Sigma}_j^{-1}$ directamente usando una rutina de inversión de propósito general como `np.linalg.inv()`. Durante la predicción, calcula explícitamente la forma cuadrática $\boldsymbol{\delta}_j^T \mathbf{\Sigma}_j^{-1} \boldsymbol{\delta}_j$.
*   **`QDA_Chol1`:** Evita la inversión directa de $\mathbf{\Sigma}_j$. En su lugar, sigue un camino numéricamente más robusto:
    1.  Calcula $\mathbf{\Sigma}_j$.
    2.  Obtiene la descomposición de Cholesky: $\mathbf{\Sigma}_j = \mathbf{L}_j \mathbf{L}_j^T$.
    3.  Calcula y almacena la inversa de la matriz triangular inferior, $\mathbf{L}_j^{-1}$.

**Paso a paso de la predicción en `QDA_Chol1`:**

1.  **Centrado:** Se calcula el vector de diferencia $\boldsymbol{\delta}_j = \mathbf{x} - \boldsymbol{\mu}_j$.
2.  **Transformación:** Se calcula el vector transformado $\mathbf{y}_j = \mathbf{L}_j^{-1} \boldsymbol{\delta}_j$.
3.  **Cálculo de la norma:** La forma cuadrática se obtiene calculando la suma de los cuadrados de los elementos de $\mathbf{y}_j$, que es $\|\mathbf{y}_j\|_2^2 = \mathbf{y}_j^T \mathbf{y}_j$. Como demostramos anteriormente, esto es matemáticamente equivalente a la forma cuadrática original.
4.  **Cálculo del determinante:** El término log-determinante también se simplifica. El determinante de una matriz triangular es el producto de sus elementos diagonales. Por lo tanto:
    $$
    \frac{1}{2} \log |\mathbf{\Sigma}_j^{-1}| = \frac{1}{2} \log |(\mathbf{L}_j^{-1})^T \mathbf{L}_j^{-1}| = \log |\mathbf{L}_j^{-1}| = \log \left( \prod_{i=1}^{p} (L_j^{-1})_{ii} \right)
    $$
    Esto evita una llamada costosa a `np.linalg.det` y la reemplaza por un producto de escalares.

En resumen, `QDA_Chol1` reemplaza operaciones matriciales complejas por operaciones con matrices triangulares, que son más rápidas y estables.

10. ¿Cuáles son las diferencias entre `QDA_Chol1`, `QDA_Chol2` y `QDA_Chol3`?


Las tres variantes utilizan la descomposición de Cholesky, pero difieren en la estrategia para calcular el vector transformado $\mathbf{y}_j = \mathbf{L}_j^{-1} \boldsymbol{\delta}_j$.

*   **`QDA_Chol1`:** Sigue un enfoque de "calcular y guardar". Durante el `fit`, calcula explícitamente la matriz inversa $\mathbf{L}_j^{-1}$ usando una función genérica (`np.linalg.inv`) y la almacena. En `predict`, realiza la multiplicación matricial $\mathbf{y}_j = \mathbf{L}_j^{-1} \boldsymbol{\delta}_j$.

*   **`QDA_Chol2`:** Adopta la práctica numéricamente más recomendada: evitar la formación explícita de la inversa. En lugar de calcular $\mathbf{y}_j = \mathbf{L}_j^{-1} \boldsymbol{\delta}_j$, resuelve el sistema de ecuaciones lineales triangulares $\mathbf{L}_j \mathbf{y}_j = \boldsymbol{\delta}_j$. Dado que $\mathbf{L}_j$ es triangular inferior, este sistema se resuelve de manera muy eficiente mediante **sustitución hacia adelante (forward substitution)**, implementada en `scipy.linalg.solve_triangular`.

*   **`QDA_Chol3`:** Es un híbrido. Al igual que `QDA_Chol1`, calcula y almacena explícitamente la inversa $\mathbf{L}_j^{-1}$. Sin embargo, en lugar de usar una rutina de inversión genérica, utiliza `dtrtri` de LAPACK, una función altamente optimizada y especializada para invertir matrices triangulares. Se espera que sea más rápida y precisa que la inversión de propósito general para este caso específico.

11. Comparar la performance de las 7 variantes de QDA implementadas hasta ahora ¿Qué se observa?¿Hay alguna de las implementaciones de `QDA_Chol` que sea claramente mejor que las demás?¿Alguna que sea peor?

In [14]:
# Benchmark de las 7 variantes con Letters dataset
X_letter, y_letter = get_letters_dataset()
y_letter_encoded = label_encode(y_letter.reshape(-1,1))

b_letter_7 = Benchmark(
    X_letter, y_letter_encoded,
    same_splits=False,
    n_runs=100,
    warmup=20,
    mem_runs=30,
    test_sz=0.2
)

to_bench = [QDA, TensorizedQDA, FasterQDA, EfficientQDA, QDA_Chol1, QDA_Chol2, QDA_Chol3]

for model in to_bench:
    b_letter_7.bench(model)

summ_letter_7 = b_letter_7.summary(baseline='QDA')
summ_letter_7[[
    'train_median_ms', 'test_median_ms', 'mean_accuracy',
    'train_speedup', 'test_speedup',
    'train_mem_reduction', 'test_mem_reduction'
]]

Benching params:
Total runs: 150
Warmup runs: 20
Peak Memory usage runs: 30
Running time runs: 100
Train size rows (approx): 16000
Test size rows (approx): 4000
Test size fraction: 0.2


QDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

...

QDA_Chol2 (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

QDA_Chol2 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

QDA_Chol3 (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

QDA_Chol3 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

,train_median_ms,test_median_ms,mean_accuracy,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,
QDA,6.02315,1397.94155,0.886117,1.000000,1.000000,1.000000,1.000000
TensorizedQDA,5.47270,278.40430,0.885303,1.100581,5.021264,1.001817,0.631634
FasterQDA,7.17515,1731.45765,0.884827,0.839446,0.807378,1.000000,0.000030
EfficientQDA,6.23735,21.77670,0.884890,0.965658,64.194371,0.998189,0.002499
QDA_Chol1,8.86180,1087.20925,0.884770,0.679676,1.285807,1.000578,1.021381
QDA_Chol2,7.43090,2954.01330,0.885433,0.810555,0.473235,0.999802,1.007427
QDA_Chol3,7.88415,985.41015,0.885807,0.763957,1.418639,0.999993,1.024227


**Análisis (Punto 11):**

El benchmark de las siete variantes introduce las implementaciones Cholesky y permite evaluar su contribución relativa.

**Tiempo de entrenamiento:** de manera llamativa, las tres variantes Cholesky son en este benchmark entre un 19% y un 32% más lentas que `QDA` en entrenamiento (speedups de 0.68×, 0.81× y 0.76× para `Chol1`, `Chol2` y `Chol3` respectivamente). Esto contrasta con la expectativa teórica de que Cholesky debería ser más rápido por su menor conteo de operaciones ($\mathcal{O}(p^3/3)$ vs. $\mathcal{O}(p^3)$). Con $p = 16$, los tiempos absolutos son muy pequeños (~6–9 ms) y dominados por el overhead de las llamadas a LAPACK y la gestión del objeto Python, lo que puede revertir la ventaja teórica. En la práctica, para $p$ pequeño la diferencia en FLOPs no justifica el costo fijo de las rutinas especializadas.

**Tiempo de predicción — variantes Cholesky vs. `QDA`:**
- `QDA_Chol3` es la mejor de las tres en test (985 ms, speedup de 1.42×), seguida de cerca por `QDA_Chol1` (1087 ms, 1.29×). La ganancia sobre `QDA` base es real pero modesta.
- `QDA_Chol2` es la peor implementación de todo el conjunto en predicción (2954 ms, speedup de 0.47×), es decir **más del doble de lenta que `QDA`**. El motivo es claro: resolver el sistema $\mathbf{L}_j \mathbf{y}_j = \boldsymbol{\delta}_j$ mediante `solve_triangular` en cada predicción individual genera $n \times k = 4000 \times 26 = 104.000$ llamadas a LAPACK, cuyo costo fijo por llamada supera ampliamente la ventaja algorítmica de la sustitución hacia adelante sobre la multiplicación matricial directa.

**¿Hay alguna claramente mejor?** En test, `QDA_Chol3` y `QDA_Chol1` son las mejores de las tres Cholesky (empate técnico), aunque muy lejos de `EfficientQDA` (21.78 ms, 64.19×). Ninguna variante Cholesky base ataca el cuello de botella real de la predicción: el bucle explícito sobre las $n$ observaciones.

**¿Hay alguna claramente peor?** `QDA_Chol2` en predicción es la peor de todas las variantes, incluyendo al `QDA` base. Para datasets de este tamaño, almacenar $\mathbf{L}^{-1}$ (como hacen `Chol1` y `Chol3`) es claramente preferible a resolver el sistema en línea observación por observación.

**Conclusión:** las variantes Cholesky base no mejoran el cuello de botella en predicción porque no eliminan el bucle sobre observaciones. La ganancia de Cholesky en entrenamiento es teóricamente sólida pero empíricamente marginal para la dimensionalidad de este dataset. El dominador absoluto en predicción sigue siendo `EfficientQDA`.

### Optimización

12. Implementar el modelo `TensorizedChol` paralelizando sobre clases/observaciones según corresponda. Se recomienda heredarlo de alguna de las implementaciones de `QDA_Chol`, aunque la elección de cuál de ellas queda a cargo del alumno según lo observado en los benchmarks de puntos anteriores.

In [15]:
# TensorizedChol hereda de QDA_Chol3 porque:
# - Chol1 y Chol3 tuvieron performance similar y ambas superaron a Chol2
# - Chol3 usa dtrtri (LAPACK, especializada para triangulares) que es
#   teóricamente mejor que LA.inv (genérica) de Chol1
#
# Como TensorizedQDA, solo tensoriza sobre las k clases.
# El for-loop sobre observaciones se mantiene (heredado de BaseBayesianClassifier).

class TensorizedChol(QDA_Chol3):

    def _fit_params(self, X, y):
        super()._fit_params(X, y)

        # Stack en tensores para operar sobre todas las clases a la vez
        self.tensor_L_inv = np.stack(self.L_invs)    # (k, p, p)
        self.tensor_means = np.stack(self.means)      # (k, p, 1)

        # Precomputar log-det: constante por clase, evita recalcularlo n×k veces
        self.log_dets = np.array([
            np.log(L_inv.diagonal().prod()) for L_inv in self.L_invs
        ])  # (k,)

    def _predict_log_conditionals(self, x):
        # Calcula log f_j(x) para las k clases a la vez (una sola observación)
        unbiased_x = x - self.tensor_means            # (k, p, 1)
        Y = self.tensor_L_inv @ unbiased_x            # (k, p, 1)
        return self.log_dets - 0.5 * (Y ** 2).sum(axis=1).flatten()  # (k,)

    def _predict_one(self, x):
        return np.argmax(self.log_a_priori + self._predict_log_conditionals(x))

13. Implementar el modelo `EfficientChol` combinando los insights de `EfficientQDA` y `TensorizedChol`. Si se desea, se puede implementar `FasterChol` como ayuda, pero no se contempla para el punto.

In [16]:
# EfficientChol combina:
# - De EfficientQDA: tensorizar sobre clases Y observaciones, evitando la matriz n×n
# - De Cholesky: la descomposición que convierte la forma cuadrática en una norma
#
# A diferencia de TensorizedChol (que solo tensoriza sobre clases y mantiene
# el for-loop sobre observaciones), EfficientChol elimina ambos for-loops.
#
# La conexión clave: en EfficientQDA usábamos
#   diag(Uᵀ Σ⁻¹ U) = sum(U * (Σ⁻¹ U), axis=1)
#
# Con Cholesky, Σ⁻¹ = (L⁻¹)ᵀ L⁻¹, entonces:
#   diag(Uᵀ (L⁻¹)ᵀ L⁻¹ U) = diag(Yᵀ Y)   donde Y = L⁻¹U
#
# Y diag(Yᵀ Y) se simplifica a:
#   sum(Y², axis=1)

class EfficientChol(TensorizedChol):
    # Hereda de TensorizedChol (que ya tiene _fit_params con tensores y log_dets)

    def predict(self, X):
        # X: (p, n) — todas las observaciones de golpe

        # Centrar: (p, n) - (k, p, 1) → (k, p, n)
        U = X - self.tensor_means

        # Y = L⁻¹ U: (k, p, p) @ (k, p, n) → (k, p, n)
        Y = self.tensor_L_inv @ U

        # diag(Yᵀ Y) = sum(Y², axis=1) → (k, n)
        dist = (Y ** 2).sum(axis=1)  # (k, n)

        # Log-posteriori: (k, 1) - (k, n) → (k, n)
        log_post = (self.log_a_priori + self.log_dets).reshape(-1, 1) - 0.5 * dist

        # Clase ganadora: (n,) → (1, n)
        return np.argmax(log_post, axis=0).reshape(1, -1)

14. Comparar la performance de las 9 variantes de QDA implementadas ¿Qué se observa? A modo de opinión ¿Se condice con lo esperado?

In [17]:
# Benchmark de las 9 variantes con Letters dataset
X_letter, y_letter = get_letters_dataset()
y_letter_encoded = label_encode(y_letter.reshape(-1,1))

b_letter_9 = Benchmark(
    X_letter, y_letter_encoded,
    same_splits=False,
    n_runs=100,
    warmup=20,
    mem_runs=30,
    test_sz=0.2
)

to_bench = [
    QDA, TensorizedQDA, FasterQDA, EfficientQDA,
    QDA_Chol1, QDA_Chol2, QDA_Chol3,
    TensorizedChol, EfficientChol
]

for model in to_bench:
    b_letter_9.bench(model)

summ_letter_9 = b_letter_9.summary(baseline='QDA')
summ_letter_9[[
    'train_median_ms', 'test_median_ms', 'mean_accuracy',
    'train_speedup', 'test_speedup',
    'train_mem_reduction', 'test_mem_reduction'
]]

Benching params:
Total runs: 150
Warmup runs: 20
Peak Memory usage runs: 30
Running time runs: 100
Train size rows (approx): 16000
Test size rows (approx): 4000
Test size fraction: 0.2


QDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

...

TensorizedChol (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

TensorizedChol (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

EfficientChol (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

EfficientChol (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

,train_median_ms,test_median_ms,mean_accuracy,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,
QDA,7.60500,1772.09590,0.886117,1.000000,1.000000,1.000000,1.000000
TensorizedQDA,7.99780,314.62770,0.885303,0.950886,5.632358,1.001817,0.637975
FasterQDA,10.44280,2260.18170,0.884827,0.728253,0.784050,1.000000,0.000031
EfficientQDA,8.97060,27.03815,0.884890,0.847769,65.540575,0.998189,0.002525
QDA_Chol1,8.39755,1004.62990,0.884770,0.905621,1.763929,1.000241,1.030872
QDA_Chol2,7.98635,2936.27825,0.885433,0.952250,0.603518,1.000128,1.022388
QDA_Chol3,8.31760,999.14625,0.885807,0.914326,1.773610,1.000074,1.032502
TensorizedChol,8.79845,111.96545,0.884995,0.864357,15.827167,0.999749,0.622483
EfficientChol,9.36165,23.80110,0.885720,0.812357,74.454370,0.999448,0.002525


**Análisis final (Punto 14):**

El benchmark completo de las nueve variantes permite leer con precisión la contribución de cada nivel de optimización y su combinación.

**Entrenamiento:** los tiempos de entrenamiento se agrupan en un rango estrecho (7.6–10.4 ms) sin un ganador claro. Las variantes tensoriales heredan `_fit_params` de `QDA` y no introducen diferencia en esta etapa. Las variantes Cholesky presentan un overhead leve respecto al baseline, atribuible al costo fijo de `dtrtri`/`LA.inv` para $p = 16$ pequeño.

**Predicción — la historia en cuatro escalones:**

1. **`QDA` base (1772 ms):** referencia. Bucle sobre $n$ observaciones × bucle sobre $k$ clases.

2. **Tensorizar sobre clases:** `TensorizedQDA` (315 ms, 5.63×) y `TensorizedChol` (112 ms, 15.83×). Ambas eliminan el bucle sobre las $k$ clases. La diferencia entre ellas —factor ~2.8×— se debe a que la representación Cholesky opera con matrices triangulares que resultan en operaciones de menor costo efectivo en la multiplicación batch $(k, p, p) \otimes (k, p, 1)$.

3. **Vectorizar sobre observaciones:** `EfficientQDA` (27 ms, 65.5×) y `EfficientChol` (24 ms, 74.5×). Ambas eliminan también el bucle sobre las $n$ observaciones. El salto de escalón respecto a las variantes solo-tensoriales es de un orden de magnitud (factor ~11.7× para `QDA` y ~4.7× para `Chol`).

4. **Combinación óptima:** `EfficientChol` es el modelo más rápido en predicción (23.8 ms, 74.5× speedup). En este entorno de ejecución, `EfficientChol` supera levemente a `EfficientQDA` (27 ms, 65.5×). Esto es coherente con que `EfficientChol` opera sobre matrices $\mathbf{Y} = \mathbf{L}^{-1}\mathbf{U}$ donde $\mathbf{L}^{-1}$ es triangular —estructura que BLAS puede aprovechar durante la multiplicación batch— mientras que `EfficientQDA` multiplica por $\boldsymbol{\Sigma}^{-1}$ densa.

**Memoria en predicción:** `FasterQDA` confirma ser la peor opción posible, consumiendo ~32.000× más RAM que el baseline ($\text{test\_mem\_reduction} \approx 3.1 \times 10^{-5}$). `EfficientQDA` y `EfficientChol` presentan el menor uso de memoria de todo el conjunto ($\approx 400\times$ menos que `QDA`), coherente con operar exclusivamente sobre estructuras $\mathcal{O}(np)$.

**Accuracy:** idéntica en las nueve variantes (0.8848–0.8861), con diferencias atribuibles únicamente a la variabilidad del split aleatorio. Dada la verificación subsiguiente, podemos observar que se confirma equivalencia matemática exacta entre implementaciones.

**Conclusión:** la cadena `QDA` → `TensorizedQDA` → `EfficientQDA` → `EfficientChol` acumula optimizaciones ortogonales. El salto más significativo en predicción lo produce la vectorización sobre observaciones con el truco Hadamard (de ~5–16× a ~65–74×), no la factorización de Cholesky. La contribución de Cholesky es real pero secundaria en este régimen: aporta ~1.14× adicional en predicción sobre `EfficientQDA` y ~2.8× en la variante tensorizada. El mensaje de diseño es claro: **antes de sofisticar el álgebra lineal, conviene eliminar los bucles explícitos sobre observaciones**.

---

## Verificación de equivalencia entre implementaciones

Antes de confiar ciegamente en los benchmarks, es necesario asegurarse de que todas las implementaciones producen **exactamente las mismas predicciones**. La siguiente celda lo verifica sobre un subset del dataset de letras.


In [18]:
# Verificacion de equivalencia matematica entre las 9 implementaciones
np.random.seed(42)
X_full_v, y_full_v = get_letters_dataset()
y_enc_v            = label_encode(y_full_v.reshape(-1, 1))

Xtr_v, Xte_v, ytr_v, yte_v = split_transpose(
    X_full_v, y_enc_v, test_size=0.1, random_state=0
)

todas = [QDA, TensorizedQDA, FasterQDA, EfficientQDA,
         QDA_Chol1, QDA_Chol2, QDA_Chol3, TensorizedChol, EfficientChol]

preds = {}
for cls in todas:
    m = cls()
    m.fit(Xtr_v, ytr_v)
    preds[cls.__name__] = m.predict(Xte_v).flatten()

ref = preds['QDA']
print("Verificacion de equivalencia (todas vs QDA):")
for name, p in preds.items():
    iguales = np.all(p == ref)
    acc     = (p == yte_v.flatten()).mean()
    print(f"  {name:25s}: {'OK' if iguales else 'DIFERENTE':8s}  (accuracy={acc:.4f})")


Verificacion de equivalencia (todas vs QDA):
  QDA                      : OK        (accuracy=0.8805)
  TensorizedQDA            : OK        (accuracy=0.8805)
  FasterQDA                : OK        (accuracy=0.8805)
  EfficientQDA             : OK        (accuracy=0.8805)
  QDA_Chol1                : OK        (accuracy=0.8805)
  QDA_Chol2                : OK        (accuracy=0.8805)
  QDA_Chol3                : OK        (accuracy=0.8805)
  TensorizedChol           : OK        (accuracy=0.8805)
  EfficientChol            : OK        (accuracy=0.8805)
